# 01 · 원자료 수집과 전처리 재현

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lab-axis/Wind-Power-Forecasting/blob/main/notebooks/01_sources_and_preprocessing.ipynb)

상명풍력 발전량 + 새별오름 AWS 883 → 시간별 병합 데이터. 이 문서는 이미 수집한 원자료의 위치·무결성·변환을 재현합니다. 자동 다운로드나 수집되지 않은 메타데이터를 만들어내지 않습니다. 모든 계산은 `src/data/*.py`, 호출 안내는 `src/workflows/research.py`에 있습니다.

**실행:** `Python (wind_power)` 커널에서 위→아래 Run All. 노트북은 각각 독립 실행할 수 있습니다. 실제 재학습은 03/04의 명시적 옵션에서만 시작합니다.

In [ ]:
from pathlib import Path
import os, sys

# Google Colab 자동 감지 및 환경 설정
if 'google.colab' in sys.modules or (os.path.exists('/content') and not (Path.cwd() / "configs/base_config.yaml").exists()):
    repo_dir = Path('/content/Wind-Power-Forecasting')
    if not repo_dir.exists():
        print("Google Colab 환경 감지: Wind-Power-Forecasting 저장소를 클론합니다...")
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/lab-axis/Wind-Power-Forecasting.git", str(repo_dir)], check=True)
        print("필수 의존 패키지를 설치합니다 (colab/requirements-colab.txt)...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(repo_dir / "colab/requirements-colab.txt")], check=True)
    os.chdir(repo_dir)
    sys.path.insert(0, str(repo_dir))
    try:
        get_ipython().run_line_magic('cd', str(repo_dir))
    except Exception:
        pass

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "configs/base_config.yaml").exists())
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
runtime = ROOT / ".experiment_archive/python_runtime"
if runtime.exists(): sys.path.insert(0, str(runtime))
from IPython.display import display
from src.workflows import research as wf
%matplotlib inline

In [ ]:
wf.environment()

## 1. 데이터 출처와 로컬 원본

- 발전량: [한국중부발전 상명풍력 발전 실적](https://www.data.go.kr/data/15119454/fileData.do). `data/raw/generation/`의 wide CSV(기준일, 1시~24시).
- 기상: [기상청 AWS](https://data.kma.go.kr/), 새별오름 883. `data/raw/weather/saebyeol_883-hour/`의 2023·2024·2025 시간 CSV.
- 분 자료의 시간자료 재구성 감사는 `src/experiments/audit_aws_reconstruction.py` 및 `reports/aws_reconstruction_validation.json`에 별도로 보존합니다. 2026 후보 기상은 본 실험에 병합하지 않습니다.

아래 SHA-256은 **현재 파일**의 식별자입니다. 다운로드 당시 제공기관 설명·정비 이력·전달 지연을 증명하는 값은 아닙니다.

In [ ]:
wf.source_inventory()

## 2. 변환 규칙과 확인되지 않은 가정

발전량 단위와 시간 집계 경계는 공식적으로 확정되지 않았습니다. 원자료의 스케일 불연속을 바탕으로 코드가 사용하는 가정을 아래에 명시합니다. 21 MWh 상한은 21 MW × 가정한 1시간 구간입니다. 기관 설명이 확인되면 가정과 데이터 버전을 재검토해야 합니다.

In [ ]:
wf.protocol_table()

## 3. 원본 → generation_hourly → weather_hourly → merged_dataset

`load_generation.build_and_save_generation_interim()` → `process_saebyeol_weather.build_and_save_master_dataset()`를 호출합니다. 24시는 다음 날 00시로 변환하고, 영발전 0은 그대로 보존합니다. 기상은 정규 시간격자에서 과거값만 최대 3시간 채우며 원래 결측·보간 여부·관측 나이를 기록합니다.

기존 데이터는 덮어쓰지 않습니다. `.experiment_archive/notebook_reproduction/`에 다시 만들고 기존 45개 열 전체와 정확히 일치하는지 검사합니다. 이 단계는 파일 전체를 변환하지만 2026년 예측·성능 평가는 하지 않습니다.

In [ ]:
wf.rebuild_preprocessing()

## 4. 연구 구간의 결측과 분할

학습: 2023~2024 상반기, 검증: 2024 하반기, 개발 평가: 2025. 아래 요약과 다음 노트북 EDA는 2025년 말까지만 사용합니다. 2025년은 여러 차례 모델 개발에 참조한 구간이므로 독립 테스트라고 부르지 않습니다.

In [ ]:
wf.split_quality()

In [ ]:
wf.weather_quality()

**다음:** [02 · 실제 모델 입력 EDA](02_model_dataset_eda.ipynb). 상세 구현: `src/data/load_generation.py`, `src/data/process_saebyeol_weather.py`, `src/data/forecast_protocol.py`.